# ChurnSense-AI - 05 Model Evaluation, SHAP & Business Impact

**Goal:** Go beyond accuracy - explain *why* customers churn, tune the decision threshold for CRM, and quantify retention value.

**Prerequisites:** `04_model_training.ipynb` (saved `models/best_model.joblib`)

| Section | Deliverable |
|---------|-------------|
| 1–2 | Load best model + test data |
| 3 | **SHAP** global & local explanations |
| 4 | **Threshold tuning** (precision/recall trade-off) |
| 5 | **Business impact** estimates |
| 6 | Export artifacts for dashboards / SQL |

## 1. Setup

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import compute_metrics, plot_confusion_matrix
from src.explain import (
    compute_shap_values,
    create_shap_explainer,
    estimate_business_impact,
    evaluate_at_threshold,
    find_threshold_balanced,
    find_threshold_by_f1,
    load_best_model,
    load_training_summary,
    plot_precision_recall_curve,
    plot_shap_beeswarm,
    plot_shap_summary_bar,
    plot_shap_waterfall,
    plot_threshold_sweep,
    save_explainability_artifacts,
    shap_mean_abs_importance,
    sweep_thresholds,
)
from src.train import load_feature_matrices

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

print(f"Project root: {PROJECT_ROOT}")

## 2. Load best model and test set

In [ ]:
model, model_name = load_best_model(MODELS_DIR)
summary = load_training_summary(MODELS_DIR)

X_train, y_train, X_test, y_test, feature_names = load_feature_matrices(PROCESSED_DIR)

print(f"Best model: {model_name}")
if summary:
    print(f"Phase 4 ROC-AUC: {summary.get('best_roc_auc', 'n/a')}")
print(f"Test set: {X_test.shape[0]:,} customers | {X_test.shape[1]} features")

In [ ]:
from src.evaluate import predict_scores

y_pred_default, y_proba = predict_scores(model, X_test)
default_metrics = compute_metrics(y_test, y_pred_default, y_proba)

print(f"Default threshold (0.5) metrics:")
for k, v in default_metrics.items():
    print(f"  {k}: {v:.4f}")

## 3. SHAP explainability (XAI)

**What is SHAP?**  
SHapley Additive exPlanations assign each feature a contribution to a prediction - grounded in game theory, model-agnostic when using generic explainers.

**Why it matters for telecom:**  
CRM leaders need to know *why* a customer is high-risk, not just a score. SHAP supports targeted offers (contract upgrade vs autopay vs support bundle).

**Paper alignment:** Chang et al. (2024) use LIME + SHAP for global and local churn explanations.

In [ ]:
explainer = create_shap_explainer(model, X_train, feature_names)
shap_values = compute_shap_values(explainer, X_test, max_samples=500)

global_importance = shap_mean_abs_importance(shap_values)
print("Top 15 features by mean |SHAP|:")
display(global_importance.head(15))

In [ ]:
plot_shap_summary_bar(shap_values, max_display=15)

In [ ]:
plot_shap_beeswarm(shap_values, max_display=15)

### Insights - global SHAP

Read the beeswarm plot:
- **Red** = higher feature value pushes toward **churn** (positive SHAP).
- **Blue** = higher value pushes toward **stay**.

**Typical top drivers (Telco):**
1. **Contract = Month-to-month** - highest churn risk  
2. **Low tenure** - early lifecycle fragility  
3. **High MonthlyCharges** - price sensitivity  
4. **Electronic check / manual pay** - friction  
5. **Fiber + no Tech Support / Online Security** - service gaps  

### How businesses use this

| SHAP signal | Retention play |
|-------------|----------------|
| High tenure SHAP (stay) | Reward loyalty, referral program |
| MTM contract SHAP (churn) | Annual plan discount |
| High charges SHAP (churn) | Plan downgrade / loyalty credit |
| No tech support SHAP (churn) | Free support trial |

### Local explanation - one high-risk customer

In [ ]:
# Explain the customer with highest predicted churn probability in the SHAP sample
sample_proba = model.predict_proba(shap_values.data)[:, 1]
high_risk_idx = int(np.argmax(sample_proba))
print(f"Sample index {high_risk_idx} | P(churn) = {sample_proba[high_risk_idx]:.3f}")
plot_shap_waterfall(shap_values, row_index=high_risk_idx)

**Local SHAP use case:** Call-center agent sees *this* customer's top 5 drivers before offering a save deal - personalized, not generic.

## 4. Threshold tuning

**Default threshold = 0.5** is not always optimal.

| Stakeholder priority | Tune toward |
|----------------------|-------------|
| Maximize **recall** (catch every churner) | Lower threshold (~0.35–0.45) |
| Maximize **precision** (cheap campaigns) | Higher threshold (~0.55–0.65) |
| Balance outreach cost vs saves | Optimize **F1** or PR curve |

**Telecom default:** Missing a churner costs more than a false coupon → favor **recall**, with a minimum precision floor.

In [ ]:
sweep_df = sweep_thresholds(y_test, y_proba)
threshold_f1 = find_threshold_by_f1(sweep_df)
threshold_balanced = find_threshold_balanced(sweep_df, min_precision=0.50)

print(f"Best F1 threshold: {threshold_f1:.2f}")
print(f"Balanced threshold (prec>=50%): {threshold_balanced:.2f}")

display(sweep_df.sort_values("f1", ascending=False).head(8))

In [ ]:
plot_threshold_sweep(sweep_df, optimal_threshold=threshold_balanced)
pr_threshold = plot_precision_recall_curve(y_test, y_proba)
print(f"PR-curve max-F1 threshold: {pr_threshold:.3f}")

In [ ]:
# Selected threshold for production scoring
SELECTED_THRESHOLD = threshold_balanced

tuned = evaluate_at_threshold(y_test, y_proba, SELECTED_THRESHOLD)
default_eval = evaluate_at_threshold(y_test, y_proba, 0.5)

comparison = pd.DataFrame([
    {"scenario": "Default (0.50)", **default_eval["metrics"], "flagged_pct": default_eval["flagged_pct"]},
    {"scenario": f"Tuned ({SELECTED_THRESHOLD:.2f})", **tuned["metrics"], "flagged_pct": tuned["flagged_pct"]},
])
display(comparison.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
from sklearn.metrics import confusion_matrix
cm_tuned = confusion_matrix(y_test, tuned["y_pred"])
plot_confusion_matrix(cm_tuned, f"{model_name} @ threshold {SELECTED_THRESHOLD:.2f}", ax=ax)
plt.tight_layout()
plt.show()

### Insights - threshold tuning

- Lowering the threshold **increases recall** (more churners caught) but **more false alarms**.
- Raising the threshold **saves campaign budget** but **misses at-risk revenue**.
- Document the chosen threshold in CRM playbooks so marketing and data science align.

## 5. Business impact analysis

Simple economics model on the **test set** (illustrative - adjust assumptions for your market).

In [ ]:
# Assumptions — edit for your business case
AVG_MONTHLY_REVENUE = 70.0      # USD, aligns with dataset median MRC
RETENTION_OFFER_COST = 15.0     # USD per outreach (call + discount)
SAVE_RATE = 0.30                  # 30% of contacted true churners retained

business_default = estimate_business_impact(
    y_test, y_proba, threshold=0.5,
    avg_monthly_revenue=AVG_MONTHLY_REVENUE,
    retention_offer_cost=RETENTION_OFFER_COST,
    save_rate_if_contacted=SAVE_RATE,
)

business_tuned = estimate_business_impact(
    y_test, y_proba, threshold=SELECTED_THRESHOLD,
    avg_monthly_revenue=AVG_MONTHLY_REVENUE,
    retention_offer_cost=RETENTION_OFFER_COST,
    save_rate_if_contacted=SAVE_RATE,
)

print("=== Default threshold 0.50 ===")
display(business_default)
print(f"\n=== Tuned threshold {SELECTED_THRESHOLD:.2f} ===")
display(business_tuned)

### Business narrative (for interviews)

1. **Problem:** Telecom churn erodes recurring revenue; acquisition costs 5–10× retention.  
2. **Solution:** ChurnSense-AI scores customers, explains drivers via SHAP, and sets a CRM threshold aligned to recall.  
3. **Impact:** More true churners contacted → estimated annual revenue saved ↑ (see table above).  
4. **Trade-off:** Outreach cost rises with lower thresholds - finance + marketing must agree on ROI.  
5. **Governance:** SHAP documents *why* a customer was flagged - supports GDPR-style transparency.

## 6. Risk segmentation for CRM

In [ ]:
risk_df = pd.DataFrame({
    "churn_probability": y_proba,
    "actual_churn": y_test,
})

risk_df["risk_band"] = pd.cut(
    risk_df["churn_probability"],
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=["Low", "Medium", "High", "Critical"],
    include_lowest=True,
)

band_summary = (
    risk_df.groupby("risk_band", observed=True)
    .agg(customers=("actual_churn", "size"), actual_churn_rate=("actual_churn", "mean"))
    .assign(actual_churn_pct=lambda x: (x["actual_churn_rate"] * 100).round(1))
)
display(band_summary)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=band_summary.reset_index(), x="risk_band", y="actual_churn_pct", palette="Reds", ax=ax)
ax.set_ylabel("Actual churn rate (%)")
ax.set_title("Churn rate by model risk band — validation")
plt.tight_layout()
plt.show()

**CRM playbook:**
- **Critical / High** → retention specialist + best offer  
- **Medium** → automated email + autopay incentive  
- **Low** → upsell / referral, no discount

## 7. Save Phase 5 artifacts

In [ ]:
threshold_meta = {
    "model": model_name,
    "default_threshold": 0.5,
    "selected_threshold": SELECTED_THRESHOLD,
    "threshold_f1_max": threshold_f1,
    "threshold_balanced": threshold_balanced,
    "metrics_at_selected": tuned["metrics"],
    "assumptions": {
        "avg_monthly_revenue": AVG_MONTHLY_REVENUE,
        "retention_offer_cost": RETENTION_OFFER_COST,
        "save_rate": SAVE_RATE,
    },
}

save_explainability_artifacts(
    global_importance,
    sweep_df,
    business_tuned,
    threshold_meta,
    PROCESSED_DIR,
)

# Save threshold for Streamlit / inference
(MODELS_DIR / "churn_threshold.json").write_text(
    json.dumps({"threshold": SELECTED_THRESHOLD, "model": model_name}, indent=2),
    encoding="utf-8",
)

print("Saved to data/processed/ and models/churn_threshold.json")

## 8. Phase 5 summary

| Artifact | Purpose |
|----------|---------|
| `shap_global_importance.csv` | Dashboard feature rankings |
| `threshold_sweep.csv` | Ops threshold sensitivity |
| `business_impact_summary.csv` | Executive ROI slide |
| `threshold_selection.json` | Documented decision |
| `models/churn_threshold.json` | Production scoring cutoff |
